In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar dataset público Lending Club Loan Data (Kaggle)
arquivo = '../data/raw/accepted_2007_to_2018Q4.csv'

# Sorteio para ler apenas 5% da base de dados e economizar memória RAM
percentual = 0.05
skip_logic = lambda i: i > 0 and np.random.rand() > percentual
print("Carregando amostra de dados...")
df = pd.read_csv(arquivo, skiprows=skip_logic, low_memory=False)

# Exploração dos dados
print(df.info())
print(df.describe())

Carregando amostra de dados...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112840 entries, 0 to 112839
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 130.0+ MB
None
       member_id      loan_amnt    funded_amnt  funded_amnt_inv  \
count        0.0  112834.000000  112834.000000    112834.000000   
mean         NaN   15072.494106   15066.491261     15047.595742   
std          NaN    9201.954367    9199.602550      9204.630832   
min          NaN     600.000000     600.000000         0.000000   
25%          NaN    8000.000000    8000.000000      8000.000000   
50%          NaN   12925.000000   12875.000000     12800.000000   
75%          NaN   20000.000000   20000.000000     20000.000000   
max          NaN   40000.000000   40000.000000     40000.000000   

            int_rate    installment    annual_inc            dti  \
count  112834.000000  112834.000000  1.128340e+05  112758.000000   
mean       13.098551     446.706767  7.784105

In [3]:
print("Iniciando a limpeza dos dados...")

# 1. Remover colunas irrelevantes
limite_minimo_preenchido = len(df) * 0.70 # Manter apenas colunas com pelo menos 70% de preenchimento
df_limpo = df.dropna(thresh=limite_minimo_preenchido, axis=1)

# 2. Filtar o status do empréstimo (A variável alvo)
status_validos = ['Fully Paid', 'Charged Off']
df_limpo = df_limpo[df_limpo['loan_status'].isin(status_validos)].copy() # A coluna 'loan_status' descreve o que aconteceu com o empréstimo

# 3. Criar a coluna inadimplente (0 ou 1)
df_limpo['inadimplente'] = (df_limpo['loan_status'] == 'Charged Off').astype(int)

# Resultado da limpeza
print("\n--- Resultados da Limpeza Inicial ---")
print(f"Colunas originais: {df.shape[1]} -> Sobraram: {df_limpo.shape[1]} colunas úteis.")
print(f"Linhas originais: {df.shape[0]} -> Sobraram: {df_limpo.shape[0]} empréstimos finalizados.")

# Quantos pagaram vs quantos deram calote
print("\n Distribuição de Inadimplência:")
print(df_limpo['inadimplente'].value_counts(normalize=True) * 100)

Iniciando a limpeza dos dados...

--- Resultados da Limpeza Inicial ---
Colunas originais: 151 -> Sobraram: 94 colunas úteis.
Linhas originais: 112840 -> Sobraram: 67413 empréstimos finalizados.

 Distribuição de Inadimplência:
inadimplente
0    80.181864
1    19.818136
Name: proportion, dtype: float64


In [4]:
print(f"Linhas antes de tratar os nulos: {df_limpo.shape[0]}")

# 1. Separar as colunas numéricas e categóricas (texto)
colunas_numericas = df_limpo.select_dtypes(include=['float64', 'int64']).columns
colunas_categoricas = df_limpo.select_dtypes(include=['object']).columns

# 2. Preencher os nulos das colunas numéricas com a mediana
for col in colunas_numericas:
    if df_limpo[col].isnull().sum() > 0:
        mediana = df_limpo[col].median()
        df_limpo[col] = df_limpo[col].fillna(mediana)

# 3. Preencher os nulos das colunas categóricas com a string "Missing" (Desconhecido)
for col in colunas_categoricas:
    if df_limpo[col].isnull().sum() > 0:
        df_limpo[col] = df_limpo[col].fillna("Missing")

# 4. Verificação final
nulos_restantes = df_limpo.isnull().sum().sum()
print(f"Total de valores nulos restantes no DataFrame: {nulos_restantes}")
print(f"Dimensões finais do dataset limpo e tratado: {df_limpo.shape}")

Linhas antes de tratar os nulos: 67413
Total de valores nulos restantes no DataFrame: 0
Dimensões finais do dataset limpo e tratado: (67413, 94)


In [5]:
# Salvar o dataset limpo em um novo arquivo CSV
df_limpo.to_csv('../data/processed/data_cleaned.csv', index=False)
print("Dados tratados salvos com sucesso!")

Dados tratados salvos com sucesso!


In [6]:
print(f"Colunas antes da seleção: {df_limpo.shape[1]}")

# 1. Remover colunas administrativas, de texto livre ou que causam vazamento de dados (Data Leakage)
colunas_para_remover = [
    'id', 'member_id', 'url', 'desc', 'title', 'emp_title',
    'zip_code', 'addr_state', 'earliest_cr_line', 'issue_d',
    'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d'
]

# Remover apenas as colunas que existem no DataFrame
colunas_existentes_para_remover = [col for col in colunas_para_remover if col in df_limpo.columns]
df_selecionado = df_limpo.drop(columns=colunas_existentes_para_remover)

# 2. Separar as features (X) da nossa variável alvo (y)
# O 'y' é o que queremos prever (inadimplente: 0 ou 1)
# O 'x' são todas as outras colunas que servirão de base para o modelo
x = df_selecionado.drop(columns=['loan_status', 'inadimplente'])
y = df_selecionado['inadimplente']

print(f"Colunas após remover metadados e IDs: {x.shape[1]}")
print(f"Total de amostras prontas para modelagem: {x.shape[0]}")

Colunas antes da seleção: 94
Colunas após remover metadados e IDs: 82
Total de amostras prontas para modelagem: 67413


In [13]:
# Salvando o dataset final para modelagem
df_selecionado.to_csv('../data/processed/dados_tratados_model.csv', index=False)
print("Dataset final para modelagem salvo com sucesso")

Dataset final para modelagem salvo com sucesso


In [ ]:
# Mapear os tipos de dados restantes (13 'object')
x.info()

# Forçar print da lista de colunas de texto
colunas_texto = x.select_dtypes(include=['object']).columns
print("\n--- Nomes das colunas de texto (object) ---")
print(colunas_texto)

<class 'pandas.core.frame.DataFrame'>
Index: 67413 entries, 0 to 112838
Data columns (total 82 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   loan_amnt                   67413 non-null  float64
 1   funded_amnt                 67413 non-null  float64
 2   funded_amnt_inv             67413 non-null  float64
 3   term                        67413 non-null  object 
 4   int_rate                    67413 non-null  float64
 5   installment                 67413 non-null  float64
 6   grade                       67413 non-null  object 
 7   sub_grade                   67413 non-null  object 
 8   emp_length                  67413 non-null  object 
 9   home_ownership              67413 non-null  object 
 10  annual_inc                  67413 non-null  float64
 11  verification_status         67413 non-null  object 
 12  pymnt_plan                  67413 non-null  object 
 13  purpose                     67413 n

In [21]:
# 1. Aplicar o One-Hot Encoding em todas as colunas de texto de x
x_enconded = pd.get_dummies(x, columns=colunas_texto, drop_first=True)

# 2. Verificar o resultado da transformação
print(f" Total de colunas antes da transformação: {x.shape[1]}")
print(f"Total de colunas depois da transformação: {x_enconded.shape[1]}")

# 3. Garantir que tudo ficou numérico (deve retornar True)
todos_numericos = all(x_enconded.dtypes != 'object')
print(f"Todas as colunas agora são númericas: {todos_numericos}")

 Total de colunas antes da transformação: 82
Total de colunas depois da transformação: 145
Todas as colunas agora são númericas: True


In [ ]:
# Divindo os dados: 80% para treino e 20% para teste
x_train, x_test, y_train, y_test = train_test_split(
    x_enconded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y # Garante que a proporção de inadimplentes seja igual no treino e no teste.
)

print("--- Divisão de Dados Concluída ---")
print(f"Linhas para treino: {x_train.shape[0]}")
print(f"Linhas para teste: {x_test.shape[0]}")

--- Divisão de Dados Concluída ---
Linhas para treino: 53930
Linhas para teste: 13483


In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

# 1.
modelo = RandomForestClassifier(n_estimators=100, max_depth=10,
                                random_state=42, n_jobs=-1)

#  2. Treinar o modelo passando x (recursos) e y (gabarito) juntos
modelo.fit(x_train, y_train)

# 3. Prever probabilidade no conjunto de teste
y_pred_proba = modelo.predict_proba(x_test)[:,1]

# Exibir o resultado da métrica
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Resultado AUC-ROC: {auc:.4f}")

Resultado AUC-ROC: 0.9996


In [27]:
# Criar uma série com a importância das colunas
importancias = pd.Series(modelo.feature_importances_, index=x_train.columns)

# Exibir as 10 mais influentes
print("--- As 10 variáveis mais influentes no modelo ---")
print(importancias.nlargest(10))

--- As 10 variáveis mais influentes no modelo ---
recoveries                 0.199314
collection_recovery_fee    0.134716
last_fico_range_low        0.114121
last_fico_range_high       0.108833
total_rec_prncp            0.104535
last_pymnt_amnt            0.068979
total_pymnt                0.050904
total_pymnt_inv            0.030763
funded_amnt_inv            0.029304
installment                0.025621
dtype: float64


In [29]:
# 1. Lista de colunas com vazamento de dados (pós=concessão)
colunas_vazamento = [
    'recoveries', 'collection_recovery_fee', 'total_pymnt', 
    'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 
    'total_rec_late_fee', 'last_pymnt_amnt', 'last_fico_range_high', 
    'last_fico_range_low', 'out_prncp', 'out_prncp_inv'
]

# 2. Filtrar o x removendo essas colunas
x_limpo = x.drop(columns =[c for c in colunas_vazamento if c in x.columns])

# 3. Refazer o One-Hot Enconding nas colunas de texto restantes
colunas_texto_restantes = x_limpo.select_dtypes(include=['object']).columns.tolist()
x_encoded_limpo = pd.get_dummies(x_limpo, columns=colunas_texto_restantes, drop_first=True)

# 4. Refazer o train/test split
x_train_r, x_test_r, y_train_r, y_test_r = train_test_split(
    x_encoded_limpo, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Treinar novo modelo e avaliar
modelo_real = RandomForestClassifier(n_estimators=100, max_depth=10, 
                                     random_state=42, n_jobs=-1)
modelo_real.fit(x_train_r, y_train_r)

y_pred_real = modelo_real.predict_proba(x_test_r)[:, 1]
auc_real = roc_auc_score(y_test_r, y_pred_real)

print(f"Resultado AUC-ROC: {auc_real:.4f}")

Resultado AUC-ROC: 0.7492


In [31]:
import joblib

# Salvar o modelo treinado na pasta models
joblib.dump(modelo_real, '../models/model_v1.pkl')

print("Modelo salvo com sucesso em models/model_v1.pkl")

Modelo salvo com sucesso em models/model_v1.pkl
